# 📂 ROTBOTS — Upload Your Footage

---

Upload your own video clips as found footage. Skip if `UPLOAD_RATIO = 0`.

---

In [ ]:
# SETUP
import json, subprocess, shutil
from pathlib import Path
from IPython.display import display, Markdown, Video
from google.colab import drive, files
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
print('✅ Setup')

In [ ]:
# LOAD PLAN
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'video_plan.json').exists()])
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
with open(SESSION_DIR/'video_plan.json') as f: plan = json.load(f)
UPLOAD_DIR = SESSION_DIR / 'upload_clips'; UPLOAD_DIR.mkdir(exist_ok=True)
needed = plan.get('upload_length', 0)
num_scenes = plan.get('num_upload_scenes', 0)
print(f'✅ Session: {SESSION_NAME}')
print(f'📂 Need {needed:.0f}s of uploaded footage ({num_scenes} scenes)')
if needed == 0: print('⚠️ UPLOAD_RATIO is 0 — skip to next notebook.')

In [ ]:
# UPLOAD
print('📂 Upload your video clips (MP4):')
uploaded = files.upload()
upload_clips = []
for fn, content in uploaded.items():
    dest = UPLOAD_DIR / fn
    with open(dest, 'wb') as f: f.write(content)
    try:
        r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(dest)],capture_output=True,text=True,timeout=10)
        dur = float(r.stdout.strip())
    except: dur = 5.0
    upload_clips.append({'path':str(dest),'filename':fn,'duration':round(dur,1)})
    print(f'   ✅ {fn} ({dur:.1f}s)')

with open(SESSION_DIR/'upload_clips.json','w') as f: json.dump({'clips':upload_clips,'total':len(upload_clips)},f,indent=2)
total_dur = sum(c['duration'] for c in upload_clips)
print(f'\n✅ {len(upload_clips)} clips, {total_dur:.0f}s total (need {needed:.0f}s)')

In [ ]:
# PREVIEW
for c in upload_clips[:4]:
    display(Markdown(f'### {c["filename"]} ({c["duration"]}s)'))
    try: display(Video(c['path'],embed=True,width=480))
    except: pass

---
*ROTBOTS Workshop — LI-MA 2026*